# Sonification of flexOPT synthetics — `src/sonification.jl`

Small demonstration of the sonification module ported from **SonifSismo.jl**, developed by NF and internship students during the summer 2026.

**Why a port and not a dependency.** SonifSismo depends on `Seis.jl`, which pins `DSP = "0.6, 0.7"`, while
flexOPT runs `DSP 0.8.4`. SonifSismo also asks for Makie 0.24 / CairoMakie 0.15, and flexOPT uses Makie 0.22 / CairoMakie 0.13.
`Seis`/`Geodesics` are also unregistered. The sonification itself never needed `Seis`: it only takes the samples and
`delta` of a trace. So `src/sonification.jl` works on **plain arrays (`signal`, `dt`)**: flexOPT synthetics,
SPECFEM traces, or anything read elsewhere. The miniSEED/JMA archive layer stays in SonifSismo.

New dependency: only `WAV.jl` (added below if missing). DSP, Makie and Statistics are already in flexOPT.

Contents
1. load OPT synthetics (`data/KirishimaElastic2D.jld2`, 5 receivers; a synthetic fallback is used if the file is absent)
2. mono sonification of one receiver + timbre/volume variants
3. the five receivers played one after the other
4. stereo mix panned by receiver position
5. (optional) convolution with a music WAV
6. waveform video with the soundtrack

In [ ]:
import Pkg

function find_flexopt_root(start_dir=pwd())
    candidates = String[]
    if haskey(ENV, "FLEXOPT_ROOT")
        push!(candidates, abspath(expanduser(ENV["FLEXOPT_ROOT"])))
    end
    directory = abspath(start_dir)
    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    for candidate in unique(candidates)
        project_file = joinpath(candidate, "Project.toml")
        source_dir = joinpath(candidate, "src")
        if isfile(project_file) && isfile(joinpath(source_dir, "commonBatchs.jl"))
            return candidate
        end
    end
    error("Cannot locate flexOPT. Start Jupyter inside the repository or set ENV[\"FLEXOPT_ROOT\"] to its absolute path.")
end

flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
@show VERSION Base.active_project()

# WAV.jl is the only new dependency of src/sonification.jl (no conflict: it only needs FileIO).
if Base.find_package("WAV") === nothing
    @info "Adding WAV.jl to the flexOPT environment (one-time)"
    Pkg.add("WAV")
end

using CairoMakie, JLD2, Statistics
CairoMakie.activate!()
include(joinpath(flexopt_root, "src", "sonification.jl"))
using .sonification

outdir = joinpath(flexopt_root, "notebooks", "tmp", "sonification")   # git-ignored
mkpath(outdir)

## 1. OPT synthetics

`data/KirishimaElastic2D.jld2` holds the OPT traces of the 2-D Kirishima benchmark (5 surface receivers at
x = −20, −10, 0, 10, 20 km, 30 s, source ≈ 1.85 Hz). `data/` is git-ignored. On a fresh clone a toy record section
(delayed, decaying Ricker wavelets) is generated instead, so the notebook always runs.

In [ ]:
datafile = joinpath(flexopt_root, "data", "KirishimaElastic2D.jld2")

function ricker(t, f0, t0)
    a = (π * f0 * (t - t0))^2
    return (1 - 2a) * exp(-a)
end

if isfile(datafile)
    t, traces, receiver_x, f0 = jldopen(datafile, "r") do f
        f["opt_time"], f["opt_traces"], f["receiver_x"], f["source_frequency"]
    end
    size(traces, 1) == length(t) || (traces = permutedims(traces))   # → nt × nreceivers
    source_label = "OPT synthetics, Kirishima 2-D benchmark"
else
    @warn "$datafile not found: using a toy record section"
    f0 = 1.85
    receiver_x = [-20e3, -10e3, 0.0, 10e3, 20e3]
    t = collect(0:0.01:30)
    traces = reduce(hcat, [begin
        r = hypot(x, 5e3)
        ricker.(t, f0, 2 + r / 3500) ./ sqrt(r) .+ 0.6 .* ricker.(t, f0 / 2, 2 + r / 2000) ./ sqrt(r)
    end for x in receiver_x])
    source_label = "toy record section"
end
t = Float64.(t); traces = Float64.(traces)
dt = sonification.sampling_interval(t)
@show size(traces) dt f0 receiver_x

fig = Figure(size=(900, 420))
ax = Axis(fig[1, 1]; xlabel="time (s)", ylabel="receiver x (km)", title=source_label)
scale = 0.45 * 10 / maximum(abs, traces)
for j in axes(traces, 2)
    lines!(ax, t, receiver_x[j] / 1e3 .+ scale .* traces[:, j]; color=:black, linewidth=1)
end
fig

## 2. One receiver, mono

`suggest_acceleration(f0; target_frequency=220)` maps the source frequency to A3. Pitch and duration are coupled
(the minimal approach, as in SonifSismo): a 30 s synthetic accelerated ×120 lasts only ≈ 0.25 s.
`resampler=:dsp` uses DSP's band-limited polyphase resampler. This matters here because the synthetics are coarsely
sampled (`dt` ≈ 0.1 s), so linear interpolation adds audible imaging.

In [ ]:
jsel = argmin(abs.(receiver_x))          # receiver above the source
acc = suggest_acceleration(f0; target_frequency=220)
trace = traces[:, jsel]

plain  = sonify_signal(t, trace; acceleration=acc, resampler=:dsp)
bright = sonify_signal(t, trace; acceleration=acc, resampler=:dsp, harmonics=(0.18, 0.08))
higher = sonify_signal(t, trace; acceleration=acc, resampler=:dsp, octave=1)
louder = sonify_signal(t, trace; acceleration=acc, resampler=:dsp, gain=4, saturation=:soft)
linear = sonify_signal(t, trace; acceleration=acc, resampler=:linear)

println("acceleration ×", round(acc; digits=1), ": ",
        round(plain.seismic_duration; digits=1), " s seismic → ",
        round(plain.audio_duration; digits=3), " s audio")
for (name, r) in pairs((; plain, bright, higher, louder, linear))
    write_sonification(joinpath(outdir, "receiver$(jsel)_$(name).wav"), r)
end
plain_player = audio_html(plain)

In [ ]:
display(audio_html(bright)); display(audio_html(higher)); display(audio_html(louder)); display(audio_html(linear))

## 3. The five receivers, one after the other

Same acceleration for every receiver. Each rendering is normalized on its own here, so amplitude decay is *not*
heard (it is in the stereo mix below).

In [ ]:
renders = [sonify_signal(t, traces[:, j]; acceleration=acc, resampler=:dsp, harmonics=(0.1, 0.05))
           for j in axes(traces, 2)]
walk = concatenate_audio(renders...; gap_seconds=0.2)
write_sonification(joinpath(outdir, "receivers_sequence.wav"), walk)
@show walk.audio_duration
audio_html(walk)

## 4. Stereo mix panned by receiver position

`sonify_stereo` uses one common normalization (relative amplitudes kept) and equal-power panning of each receiver by its x.
Listen on headphones: the wavefront sweeps from the centre outwards to both sides.

In [ ]:
stereo = sonify_stereo(t, traces; positions=receiver_x, acceleration=acc, resampler=:dsp,
                       harmonics=(0.1, 0.05), gain=3, saturation=:soft)
@show size(stereo.signal) stereo.pan
write_sonification(joinpath(outdir, "receivers_stereo.wav"), stereo)
audio_html(stereo)

## 5. (optional) Convolution with a music excerpt

Set `music_path` to any WAV file. The first `kernel_seconds` are used as a convolution kernel. A measured impulse response
gives a reverb, and a music excerpt gives a smeared hybrid texture.

In [ ]:
music_path = nothing   # e.g. "/Users/nobuaki/Music/excerpt.wav"
if music_path !== nothing
    hybrid = convolve_with_audio(stereo, music_path; kernel_seconds=1.5, mix=0.3)
    write_sonification(joinpath(outdir, "receivers_stereo_music_convolution.wav"), hybrid)
    display(audio_html(hybrid))
end

## 6. Waveform video with soundtrack

At ×120 the video would last a quarter of a second. Here a lower acceleration (×30, fundamental ≈ 55 Hz) is combined
with strong harmonics and soft saturation, which put audible energy at 110 and 165 Hz and give about a 1 s clip.
For longer real records (SonifSismo/miniSEED) keep the higher acceleration.

In [ ]:
acc_video = 30
stereo_video = sonify_stereo(t, traces; positions=receiver_x, acceleration=acc_video, resampler=:dsp,
                             harmonics=(0.35, 0.25), gain=4, saturation=:soft)
video = waveform_video(t, traces, stereo_video, joinpath(outdir, "receivers_stereo.mp4");
                       labels=["x = $(round(Int, x / 1e3)) km" for x in receiver_x],
                       framerate=30, title="flexOPT synthetics, sonified (×$(acc_video), stereo by receiver x)")
@show video stereo_video.audio_duration
audio_html(stereo_video)